In [33]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("ARTICLES_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()



# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    collection = db[ARTICLES_COLLECTION_NAME]

    # Verify connection
    client.admin.command('ping')
    print("✅ Connected to MongoDB Atlas!")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas!


In [36]:
# ---- CONFIGURATION ----
start_index = 0
end_index = 50
date_format = "%d-%m-%Y"
batch_size = end_index - start_index + 1

print(f"🔍 Starting update from index {start_index} to {end_index} (batch size: {batch_size})")

# ---- FETCH DOCUMENTS ----
print("📦 Fetching documents with 'meta.date' as string...")
docs_cursor = collection.find(
    {"meta.date": {"$type": "string"}}
).sort("_id", 1).skip(start_index).limit(batch_size)

bulk_operations = []
print("list length",docs_cursor)
count = 0


🔍 Starting update from index 0 to 50 (batch size: 51)
📦 Fetching documents with 'meta.date' as string...
list length <pymongo.cursor.Cursor object at 0x10b60f4d0>


In [37]:
from datetime import datetime, timezone

print(f"🔍 Starting per-document update using offset pagination: {start_index} to {end_index}")

updated_count = 0
skipped_count = 0

for offset in range(start_index, end_index + 1):
    try:
        doc = collection.find(
            {"meta.date": {"$type": "string"}}
        ).sort("_id", 1).skip(offset).limit(1).next()

        doc_id = doc["_id"]
        raw_date = doc["meta"]["date"]
        parsed_date = datetime.strptime(raw_date, date_format).replace(tzinfo=timezone.utc)

        result = collection.update_one(
            {"_id": doc_id},
            {"$set": {"meta.date": parsed_date}}
        )

        if result.modified_count > 0:
            print(f"✅ Updated doc ID {doc_id} (date: {raw_date} → {parsed_date})")
            updated_count += 1
        else:
            print(f"ℹ️ No changes made for doc ID {doc_id} (already up to date?)")

    except StopIteration:
        print(f"❌ No more documents found at offset {offset}. Ending early.")
        break
    except Exception as e:
        print(f"⚠️ Error at offset {offset}: {e}")
        skipped_count += 1

# ---- SUMMARY ----
print(f"\n📊 Summary:")
print(f"   ✅ Successfully updated: {updated_count}")
print(f"   ⚠️ Skipped due to errors: {skipped_count}")

🔍 Starting per-document update using offset pagination: 0 to 50
✅ Updated doc ID 67b1e41f0554959fd725edf7 (date: 05-03-2018 → 2018-03-05 00:00:00+00:00)
✅ Updated doc ID 67b1e4200554959fd725edfb (date: 05-02-2018 → 2018-02-05 00:00:00+00:00)
✅ Updated doc ID 67b1e4220554959fd725edff (date: 14-02-2018 → 2018-02-14 00:00:00+00:00)
✅ Updated doc ID 67b1e4240554959fd725ee03 (date: 12-12-2017 → 2017-12-12 00:00:00+00:00)
✅ Updated doc ID 67b1e4250554959fd725ee07 (date: 26-01-2018 → 2018-01-26 00:00:00+00:00)
✅ Updated doc ID 67b1e4270554959fd725ee0b (date: 06-01-2018 → 2018-01-06 00:00:00+00:00)
✅ Updated doc ID 67b1e4290554959fd725ee0f (date: 12-03-2018 → 2018-03-12 00:00:00+00:00)
✅ Updated doc ID 67b1e42a0554959fd725ee13 (date: 11-02-2018 → 2018-02-11 00:00:00+00:00)
✅ Updated doc ID 67b1e42c0554959fd725ee17 (date: 17-01-2018 → 2018-01-17 00:00:00+00:00)
✅ Updated doc ID 67b1e42d0554959fd725ee1b (date: 28-03-2018 → 2018-03-28 00:00:00+00:00)
✅ Updated doc ID 67b1e42f0554959fd725ee1f (dat

In [16]:
if bulk_operations:
    result = collection.bulk_write(bulk_operations)
    print(f"✅ Updated {result.modified_count} documents from index {start_index} to {end_index}.")
else:
    print("ℹ️ No updates performed.")

ℹ️ No updates performed.


In [21]:
# Check total count first
total_matching = collection.count_documents({"meta.date": {"$type": "string"}})
print(f"📦 Found {total_matching} documents with string meta.date.")

📦 Found 0 documents with string meta.date.


In [29]:
sample_cursor = collection.find(
    {"meta.date": {"$exists": True}},
    {"_id": 1, "meta": 1}
).limit(5)

In [28]:
import json
from bson import json_util
for doc in sample_cursor:
    print(json.dumps(doc, indent=2, default=json_util.default))

{'version': 'fbe7835784bd15c9809199c5c08f37e6074cb371', 'branch': 'HEAD', 'dcpClient': {'version': '4.4.21', 'resolved': 'https://registry.npmjs.org/dcp-client/-/dcp-client-4.4.21.tgz', 'integrity': 'sha512-8HN/KdOlUrPPW+lpN8WIad1uN9wD25oEJCBSvoqo2t6C1kpNbhojnff5bu1WgYuD5pBnkjWbiIBL0D+KpI66Tw==', 'dependencies': {'atob': '2.1.2', 'bravojs': '^1.0.16', 'btoa': '^1.2.1', 'chalk': '^4.1.2', 'ethereumjs-util': '^7.1.5', 'ethereumjs-wallet': '^1.0.2', 'html-to-text': '^5.1.1', 'http-proxy-agent': '^4.0.1', 'https-agent': '^1.0.0', 'https-proxy-agent': '^5.0.0', 'kvin': '1.2.14', 'nanoid': '3.3.8', 'physical-cpu-count': '^2.0.0', 'polyfill-crypto.getrandomvalues': '^1.0.0', 'regedit': '^3.0.3', 'semver': '^7.3.5', 'socket.io-client': '4.8.0', 'source-map-support': '0.5.21', 'webpack': '5.94.0', 'webpack-cli': '^4.7.2', 'xmlhttprequest-ssl': '3.0.0', 'yargs': '16.2.0'}, 'engines': {'node': '>=18', 'npm': '>=8'}, 'author': 'Distributive', 'license': 'MIT', 'dcp': {'version': '4.4.13'}}, 'built